# Phases 5–8 — Full pipeline and dashboard

Detect → filter → characterise against AIS → cross-check GFW → enrich → score →
map. Produces `outputs/events.geojson`, which is what the web app serves.

No GPU needed: YOLO nano on CPU is fine for inference over a handful of scenes.


In [ ]:
# Clone (or pull) and install. Re-run this cell after every `git push`.
import os, pathlib
if pathlib.Path('smugglers').exists():
    !cd smugglers && git pull --ff-only
else:
    !git clone https://github.com/Jaswanth-K1210/smugglers.git
%cd /content/smugglers
!pip install -q -r requirements.txt
print()
!git log --oneline -1


In [ ]:
# Credentials. Only GFW needs one — SAR and AIS are open.
# Colab: use the key icon in the left sidebar, add GFW_API_TOKEN, then run this.
from google.colab import userdata
import pathlib
tok = ""
try:
    tok = userdata.get('GFW_API_TOKEN')
except Exception:
    print("No Colab secret found. Paste your token below if you want GFW layers.")
pathlib.Path('.env').write_text(
    f"GFW_API_TOKEN={tok}\nAOI_BBOX=9.5,56.5,12.5,58.5\n")
from src.config import GFW_API_TOKEN, bbox
print("GFW token set:", bool(GFW_API_TOKEN), "| AOI:", bbox())


In [ ]:
# Weights from Phase 4, if you have them.
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p models/darksts/weights
!cp /content/drive/MyDrive/darksts/models/best.pt models/darksts/weights/ 2>/dev/null \
   && echo "weights restored" || echo "no weights — pipeline runs its AIS fallback (smoke test only)"


## Sanctioned zones (optional)

Without a GeoJSON that scoring term is 0 for every event, which is honest:
absent a zone list, proximity to one is unknown, and unknown must not inflate a
score.

In [ ]:
import json, pathlib
zones = {"type":"FeatureCollection","features":[
  {"type":"Feature","properties":{"name":"example zone"},
   "geometry":{"type":"Point","coordinates":[10.70, 57.68]}}]}
pathlib.Path('data/zones.geojson').write_text(json.dumps(zones))
print('replace this with real zones before reporting any sanctioned-proximity result')


In [ ]:
from src.run_pipeline import run
events = run('2025-06-06', '2025-06-08', max_scenes=3, zones_path='data/zones.geojson')
events.head(20)


In [ ]:
from src import dashboard
dashboard.build(events)
from IPython.display import IFrame
IFrame('outputs/dashboard/index.html', width='100%', height=560)


## Commit the results so the web app can serve them

In [ ]:
!git config user.email "koppisettyjaswanth@gmail.com" && git config user.name "Jaswanth-K1210"
!git add -f outputs/events.geojson outputs/events.csv
!git commit -m "Pipeline results for the demo date range" || echo "nothing to commit"
print("\nNow: git push from here (needs a token), or download outputs/events.geojson and push locally.")
